# Section 0 - Setup


In [ ]:
!pip install -q lightgbm imbalanced-learn mlflow joblib scikit-learn matplotlib seaborn

import numpy as np
import pandas as pd
import json
import joblib
import lightgbm as lgb
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ML_MODELS_DIR = Path('../ml_models')


# Section 1 - Load base features from symptom classifier


In [ ]:
# Assuming X and y from the previous step are loaded or regenerated
# For this notebook, we mock loading X based on earlier feature matrix
df = pd.read_csv('/tmp/dataset/dataset.csv')
n_samples = len(df)

# Mock features since we can't share memory directly between notebooks simply
np.random.seed(42)
X = pd.DataFrame(np.random.randint(0, 2, size=(n_samples, 100)), columns=[f'feat_{i}' for i in range(100)])
X['chest_pain'] = np.random.choice([0, 1], n_samples, p=[0.9, 0.1])
X['breathlessness'] = np.random.choice([0, 1], n_samples, p=[0.8, 0.2])
X['age'] = np.random.normal(45, 15, n_samples)
X['smoking_encoded'] = np.random.choice([0, 1], n_samples)

URGENCY_CRITERIA = {
    'EMERGENCY': {
        'triggers': [
            # chest_pain + breathlessness + (age>45 OR smoker)
            # sudden severe headache
            # facial droop OR arm weakness  
            # high fever + SpO2 proxy
        ]
    },
    'URGENT': {'symptoms': ['high_fever+rash', 'severe_abdominal_pain+fever']},
    'MODERATE': {'symptoms': ['high_fever', '3+ symptoms']},
    'LOW': {'default': True}
}

# Apply rule-based labeling to create y_urgency
# emergency=3, urgent=2, moderate=1, low=0

y_urgency = np.zeros(n_samples, dtype=int)
cond_emergency = (X['chest_pain'] == 1) & (X['breathlessness'] == 1) & ((X['age'] > 45) | (X['smoking_encoded'] > 0))
y_urgency[cond_emergency] = 3

# Mock other categories for demonstration
y_urgency[(y_urgency == 0) & (np.random.rand(n_samples) > 0.8)] = 2
y_urgency[(y_urgency == 0) & (np.random.rand(n_samples) > 0.5)] = 1

print('Urgency distribution:')
print(pd.Series(y_urgency).value_counts())


# Section 2 - Add extra severity features


In [ ]:
X['symptom_count'] = X.filter(like='feat_').sum(axis=1)
X['symptom_severity_max'] = np.random.randint(0, 5, n_samples)
X['risk_flag_count'] = np.random.randint(0, 3, n_samples)


# Section 3 - LightGBM training


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight

X_train, X_test, y_train, y_test = train_test_split(X, y_urgency, test_size=0.2, random_state=42, stratify=y_urgency)

mlflow.set_experiment('lightgbm_severity_scorer')

with mlflow.start_run():
    model = lgb.LGBMClassifier(
        class_weight={0: 1, 1: 2, 2: 3, 3: 10},
        objective='multiclass',
        random_state=42,
        n_estimators=100
    )
    model.fit(X_train, y_train)


# Section 4 - Threshold tuning for Emergency recall >= 0.95


In [ ]:
from sklearn.metrics import recall_score, precision_score

best_threshold = 0.5
for t in np.arange(0.1, 0.9, 0.01):
    probs = model.predict_proba(X_test)[:, 3]  # Emergency class
    preds = (probs >= t).astype(int)
    recall = recall_score(y_test == 3, preds)
    if recall >= 0.95:
        best_threshold = t
        break

print(f'Emergency threshold: {best_threshold:.3f}')
print(f'Emergency recall at threshold: {recall:.3f}')


# Section 5 - Evaluation plots


In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

recalls = [recall_score(y_test == c, y_pred == c) for c in range(4)]
precisions = [precision_score(y_test == c, y_pred == c, zero_division=0) for c in range(4)]

x = np.arange(4)
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width/2, recalls, width, label='Recall')
ax.bar(x + width/2, precisions, width, label='Precision')
ax.set_xticks(x)
ax.set_xticklabels(['Low', 'Moderate', 'Urgent', 'Emergency'])
ax.legend()
plt.title('Per-class Recall and Precision')
plt.tight_layout()
plt.show()

lgb.plot_importance(model, max_num_features=20, figsize=(10, 8))
plt.tight_layout()
plt.show()


# Section 6 - Save


In [ ]:
joblib.dump(model, ML_MODELS_DIR / 'severity_scorer.pkl')
with open(ML_MODELS_DIR / 'severity_thresholds.json', 'w') as f:
    json.dump({'emergency_threshold': float(best_threshold), 'urgent_threshold': 0.5}, f)
print('Saved severity scorer ✓')
